# Loss Landscape and Contour Explorer
Move the plane level and see the matching contour level update.

### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def L(t0,t1):
    return 0.15*(t0*t0+t1*t1) + 0.7*np.sin(t0)*np.cos(t1) + 0.25*np.sin(1.5*t0+0.3*t1)

def grad(t0,t1):
    g0 = 0.3*t0 + 0.7*np.cos(t0)*np.cos(t1) + 0.375*np.cos(1.5*t0+0.3*t1)
    g1 = 0.3*t1 - 0.7*np.sin(t0)*np.sin(t1) + 0.075*np.cos(1.5*t0+0.3*t1)
    return g0, g1

lo, hi = -5, 5
xs = np.linspace(lo, hi, 70)
ys = np.linspace(lo, hi, 70)
X, Y = np.meshgrid(xs, ys)
Z = L(X, Y)

out = widgets.Output()
plane = widgets.FloatSlider(description='plane', min=-1.0, max=6.0, step=0.05, value=1.2, readout_format='.2f', continuous_update=False)
alpha = widgets.FloatSlider(description='alpha', min=0.005, max=0.2, step=0.005, value=0.05, readout_format='.3f', continuous_update=False)
max_iter = widgets.IntSlider(description='max iter', min=1, max=200, step=1, value=40, continuous_update=False)

def gd_path(a, steps):
    t0, t1 = 3.2, -3.2
    px, py, pz = [t0], [t1], [L(t0,t1)]
    for _ in range(steps):
        g0, g1 = grad(t0,t1)
        t0 -= a*g0
        t1 -= a*g1
        px.append(t0); py.append(t1); pz.append(L(t0,t1))
    return np.array(px), np.array(py), np.array(pz)

def render(*_):
    px, py, pz = gd_path(alpha.value, max_iter.value)

    surf = go.Figure()
    surf.add_surface(x=xs, y=ys, z=Z, colorscale='Viridis', showscale=False, opacity=0.95)
    surf.add_surface(x=xs, y=ys, z=np.full_like(Z, plane.value), showscale=False, opacity=0.35,
                     colorscale=[[0,'#ef4444'],[1,'#ef4444']])
    surf.add_scatter3d(x=px, y=py, z=pz, mode='lines+markers', line=dict(color='#111', width=4), marker=dict(size=3, color='#111'))
    surf.update_layout(title='Loss landscape', scene=dict(xaxis_title='θ0', yaxis_title='θ1', zaxis_title='L(θ)',
                     xaxis=dict(range=[lo,hi]), yaxis=dict(range=[lo,hi]), zaxis=dict(range=[-1,6])), height=500)

    cont = go.Figure()
    cont.add_contour(x=xs, y=ys, z=Z, colorscale='Viridis', showscale=False, contours=dict(coloring='lines'))
    cont.add_contour(x=xs, y=ys, z=Z, showscale=False, line=dict(color='#ef4444', width=3),
                     contours=dict(start=plane.value, end=plane.value, size=1, coloring='lines'))
    cont.add_scatter(x=px, y=py, mode='lines+markers', line=dict(color='#111', width=2), marker=dict(size=4, color='#111'))
    cont.update_layout(title='Contour plot with selected level', xaxis=dict(title='θ0', range=[lo,hi]),
                      yaxis=dict(title='θ1', range=[lo,hi], scaleanchor='x'), height=500)

    with out:
        out.clear_output(wait=True)
        display(widgets.HBox([surf, cont]))

    print(f'level={plane.value:.2f} | alpha={alpha.value:.3f} | iter={max_iter.value} | final loss={pz[-1]:.3f}')

plane.observe(render, 'value')
alpha.observe(render, 'value')
max_iter.observe(render, 'value')
display(widgets.HBox([plane, alpha, max_iter]))
display(out)
render()
